# Reviewer Agent using LangGraph

**Objective:** Build a multi-step AI agent with **LangGraph** in which a **Writer** agent produces a draft, a **Reviewer** agent critiques it against a rubric, and the draft is sent back for revision until the reviewer approves it (or a maximum number of rounds is reached).

This is the *reflection / review loop* pattern.

```
          +-----------+        +-------------+   approved or max rounds   +-----------+
 START -> |  writer   | -----> |  reviewer   | -------------------------> | finalize  | -> END
          +-----------+        +-------------+                            +-----------+
                ^                     |
                |   not approved: feedback goes back to the writer
                +---------------------+
```

| LangGraph concept | In this project |
|---|---|
| **State** | shared dictionary: task, draft, feedback, review history, iteration, approved |
| **Nodes** | `writer`, `reviewer`, `finalize` (Python functions that call the LLM) |
| **Edges** | `START -> writer -> reviewer`, `finalize -> END` |
| **Conditional edge** | after `reviewer`: go back to `writer` (revise) or to `finalize` |

**Name:** ______________________  **Roll No:** ______________________

## 1. Configuration
Choose which LLM provider to use. Any one API key is enough (Groq and Gemini both have free tiers).

| PROVIDER | Key name | Where to get a key |
|---|---|---|
| `"groq"` | GROQ_API_KEY | console.groq.com |
| `"gemini"` | GOOGLE_API_KEY | aistudio.google.com |
| `"openai"` | OPENAI_API_KEY | platform.openai.com |
| `"anthropic"` | ANTHROPIC_API_KEY | console.anthropic.com |

In Colab, the best way is to add the key under the **Secrets** (key icon) panel on the left, with the key name from the table, and turn on "Notebook access".

In [ ]:
PROVIDER = "groq"          # "groq" | "gemini" | "openai" | "anthropic"
MODEL = None               # None = use the default model for the provider (change it if you get a "model not found" error)

MAX_ITERATIONS = 3         # maximum number of write -> review rounds
APPROVAL_SCORE = 8         # reviewer must give at least this score (1-10) to approve

In [ ]:
import os, sys, subprocess, getpass

PROVIDERS = {
    #  provider   : (pip package,              default model,              env var)
    "groq":      ("langchain-groq",           "llama-3.3-70b-versatile",  "GROQ_API_KEY"),
    "gemini":    ("langchain-google-genai",   "gemini-2.5-flash",         "GOOGLE_API_KEY"),
    "openai":    ("langchain-openai",         "gpt-4o-mini",              "OPENAI_API_KEY"),
    "anthropic": ("langchain-anthropic",      "claude-haiku-5-5",         "ANTHROPIC_API_KEY"),
}
pip_pkg, default_model, env_var = PROVIDERS[PROVIDER]
MODEL = MODEL or default_model

# install LangGraph + the chosen provider's package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph", "langchain-core", pip_pkg], check=True)

# API key: Colab Secrets -> environment variable -> ask
if not os.environ.get(env_var):
    try:
        from google.colab import userdata
        os.environ[env_var] = userdata.get(env_var)
    except Exception:
        os.environ[env_var] = getpass.getpass(f"Enter your {env_var}: ")

print(f"Provider: {PROVIDER} | Model: {MODEL}")

## 2. LLM, State and Structured Review Output

In [ ]:
from typing import TypedDict, List
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

def get_llm(temperature=0.3):
    if PROVIDER == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODEL, temperature=temperature)
    if PROVIDER == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODEL, temperature=temperature)
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL, temperature=temperature)
    if PROVIDER == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=MODEL, temperature=temperature)
    raise ValueError("Unknown PROVIDER")

llm = get_llm()

def text_of(response):
    """Get plain text from an LLM response (some providers return a list of parts)."""
    c = response.content
    if isinstance(c, str):
        return c
    return "".join(p if isinstance(p, str) else p.get("text", "") for p in c)


# ---- the shared state that flows through the graph ----
class AgentState(TypedDict):
    task: str            # what the user wants written
    task_type: str       # "code" or "text" (chooses the review rubric)
    draft: str           # latest draft from the writer
    feedback: str        # latest feedback from the reviewer (empty on first round)
    history: List[dict]  # one entry per review round (score, issues, ...)
    iteration: int       # number of drafts written so far
    approved: bool       # did the reviewer approve the latest draft?
    status: str          # final status message
    final_output: str    # final approved (or last) draft


# ---- structured output the reviewer must return ----
class ReviewResult(BaseModel):
    score: int = Field(description="Overall quality score from 1 (poor) to 10 (excellent)")
    approved: bool = Field(description="True only if the draft fully meets the task and the rubric")
    strengths: List[str] = Field(description="What the draft does well")
    issues: List[str] = Field(description="Specific problems that must be fixed")
    suggestions: List[str] = Field(description="Concrete, actionable improvements")

## 3. Rubrics and Prompts
The reviewer judges the draft against a rubric that depends on the task type.

In [ ]:
RUBRICS = {
    "code": [
        "Correctness: does the code solve the task, including edge cases?",
        "Readability: clear names, helpful comments, PEP 8 style",
        "Documentation: docstring with arguments, return value and an example",
        "Error handling: invalid input is handled sensibly",
        "Efficiency: no needlessly slow or complicated approach",
    ],
    "text": [
        "Relevance: fully answers the task",
        "Structure: clear introduction, body and conclusion",
        "Clarity: simple, precise language without repetition",
        "Accuracy: facts are correct and not exaggerated",
        "Style: consistent tone and good grammar",
    ],
}

WRITER_SYSTEM = {
    "code": "You are an expert Python developer. Return ONLY the complete code in one ```python block, with no extra explanation.",
    "text": "You are an expert writer. Return ONLY the final text, with no preface or explanation.",
}

REVIEWER_SYSTEM = (
    "You are a strict but fair reviewer. Judge the draft ONLY against the task and the rubric. "
    "Give a score from 1 to 10. Approve only if the draft is truly good (no important issues). "
    "Be specific: every issue must be something the writer can fix."
)

## 4. Nodes and Routing Logic
Each node receives the current **state** and returns only the keys it wants to update.

In [ ]:
def writer_node(state: AgentState) -> dict:
    """Writes the first draft, or revises the draft using the reviewer's feedback."""
    if state["feedback"]:
        user_msg = (
            f"TASK:\n{state['task']}\n\n"
            f"YOUR PREVIOUS DRAFT:\n{state['draft']}\n\n"
            f"REVIEWER FEEDBACK:\n{state['feedback']}\n\n"
            "Rewrite the draft and fix every issue raised by the reviewer."
        )
    else:
        user_msg = f"TASK:\n{state['task']}"
    response = llm.invoke([("system", WRITER_SYSTEM[state["task_type"]]), ("human", user_msg)])
    return {"draft": text_of(response), "iteration": state["iteration"] + 1}


def reviewer_node(state: AgentState) -> dict:
    """Reviews the draft and returns a structured verdict."""
    rubric = "\n".join(f"- {r}" for r in RUBRICS[state["task_type"]])
    user_msg = f"TASK:\n{state['task']}\n\nRUBRIC:\n{rubric}\n\nDRAFT TO REVIEW:\n{state['draft']}"
    reviewer = llm.with_structured_output(ReviewResult)
    result = reviewer.invoke([("system", REVIEWER_SYSTEM), ("human", user_msg)])

    # deterministic guard: approval needs BOTH the LLM's approval and a high enough score
    approved = bool(result.approved) and result.score >= APPROVAL_SCORE

    feedback = ""
    if not approved:
        feedback = "Issues:\n" + "\n".join(f"- {i}" for i in result.issues) + \
                   "\n\nSuggestions:\n" + "\n".join(f"- {s}" for s in result.suggestions)

    entry = {
        "round": state["iteration"],
        "score": result.score,
        "approved": approved,
        "strengths": result.strengths,
        "issues": result.issues,
        "suggestions": result.suggestions,
    }
    return {"approved": approved, "feedback": feedback, "history": state["history"] + [entry]}


def finalize_node(state: AgentState) -> dict:
    """Stores the final output and a status message."""
    if state["approved"]:
        status = f"Approved by the reviewer after {state['iteration']} round(s)."
    else:
        status = f"Maximum of {MAX_ITERATIONS} rounds reached - returning the latest draft (not approved)."
    return {"final_output": state["draft"], "status": status}


def route_after_review(state: AgentState) -> str:
    """Conditional edge: decide what happens after a review."""
    if state["approved"] or state["iteration"] >= MAX_ITERATIONS:
        return "finish"
    return "revise"

## 5. Build the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)

graph.add_node("writer", writer_node)
graph.add_node("reviewer", reviewer_node)
graph.add_node("finalize", finalize_node)

graph.add_edge(START, "writer")
graph.add_edge("writer", "reviewer")
graph.add_conditional_edges("reviewer", route_after_review, {"revise": "writer", "finish": "finalize"})
graph.add_edge("finalize", END)

app = graph.compile()

print(app.get_graph().draw_mermaid())      # text version of the graph (works offline)
try:
    from IPython.display import Image
    display(Image(app.get_graph().draw_mermaid_png()))   # picture version (needs internet)
except Exception as e:
    print("(Graph image not shown:", type(e).__name__, ")")

## 6. Run the Agent
`run_agent()` streams the graph so you can see every step: what the writer wrote and what the reviewer decided.

In [ ]:
def run_agent(task: str, task_type: str = "text"):
    state = {"task": task, "task_type": task_type, "draft": "", "feedback": "",
             "history": [], "iteration": 0, "approved": False, "status": "", "final_output": ""}
    final = dict(state)

    print("=" * 70)
    print("TASK:", task)
    print("=" * 70)
    for step in app.stream(state, stream_mode="updates"):
        for node, update in step.items():
            final.update(update)
            if node == "writer":
                kind = "first draft" if final["iteration"] == 1 else "revised draft"
                print(f"\n[writer]   wrote {kind} (round {update['iteration']})")
            elif node == "reviewer":
                h = update["history"][-1]
                print(f"[reviewer] score = {h['score']}/10 | approved = {h['approved']}")
                for issue in h["issues"]:
                    print(f"            - issue: {issue}")
            elif node == "finalize":
                print(f"\n[finalize] {update['status']}")
    return final

### Example 1: Code review loop

In [ ]:
result_code = run_agent(
    "Write a Python function `is_palindrome(text)` that checks whether a string is a palindrome, "
    "ignoring case, spaces and punctuation.",
    task_type="code",
)
display(Markdown("### Final output\n\n" + result_code["final_output"]))

### Example 2: Text review loop

In [ ]:
result_text = run_agent(
    "Write a short paragraph (about 120 words) explaining what an AI agent is, for a first-year university student.",
    task_type="text",
)
display(Markdown("### Final output\n\n" + result_text["final_output"]))

## 7. Review History (how the draft improved)

In [ ]:
import pandas as pd

def history_table(result):
    return pd.DataFrame([{"Round": h["round"], "Score": h["score"], "Approved": h["approved"],
                          "No. of issues": len(h["issues"])} for h in result["history"]])

print("Example 1 (code):", result_code["status"])
display(history_table(result_code))
print("\nExample 2 (text):", result_text["status"])
display(history_table(result_text))

# Full details of every review round for Example 1
for h in result_code["history"]:
    print(f"\n--- Round {h['round']}: score {h['score']}/10, approved = {h['approved']} ---")
    print("Strengths  :", "; ".join(h["strengths"]))
    print("Issues     :", "; ".join(h["issues"]) or "none")
    print("Suggestions:", "; ".join(h["suggestions"]) or "none")

In [ ]:
# Save a report of the runs (optional, handy for submission)
with open("reviewer_agent_report.md", "w") as f:
    for title, res in [("Example 1 - Code", result_code), ("Example 2 - Text", result_text)]:
        f.write(f"# {title}\n\n**Task:** {res['task']}\n\n**Status:** {res['status']}\n\n")
        for h in res["history"]:
            f.write(f"- Round {h['round']}: score {h['score']}/10, approved={h['approved']}, issues={len(h['issues'])}\n")
        f.write(f"\n## Final output\n\n{res['final_output']}\n\n")
print("Saved: reviewer_agent_report.md")

## 8. How it Works (summary for the viva)
1. **State (`AgentState`)** is the memory shared by all nodes. Each node returns only the keys it changes and LangGraph merges them into the state.
2. **writer node** calls the LLM to write a draft; if there is reviewer feedback in the state it revises the previous draft instead.
3. **reviewer node** asks the LLM to judge the draft against a rubric and return a *structured* result (`ReviewResult`: score, approved, issues, suggestions) using `with_structured_output`. A guard in code requires `score >= APPROVAL_SCORE` before the draft counts as approved.
4. **Conditional edge (`route_after_review`)** makes the graph a loop: not approved -> back to the writer; approved *or* `MAX_ITERATIONS` reached -> `finalize`. The maximum prevents an infinite loop.
5. **finalize node** stores the final output and a status message.

**Possible extensions:** add a human-in-the-loop approval step, run the generated code in a sandbox as an extra "tester" node, use a checkpointer (`MemorySaver`) for memory, or use a different LLM for the reviewer than for the writer.